# YOLO Finetuning for Zalo AI Challenge (Kaggle-compatible)

This notebook is designed to run on **Kaggle** to finetune a YOLO model using drone-style datasets such as **VisDrone** or **SeaDronesSee**.

## Usage on Kaggle
1. Create a Kaggle Notebook.
2. Attach one or more datasets, e.g.:
   - A VisDrone/SeaDronesSee dataset that is already in **YOLO format** (or with images + labels you know the paths to).
   - (Optional) A dataset containing your current pretrained weights (e.g. `yolov8n.pt` or `yolo11n.pt`) so you can start from them.
3. Adjust the paths in the configuration cell below to match your attached datasets.
4. Run training.
5. Download the resulting `best.pt` from `/kaggle/working/` and place it into your local `model/` folder to use with `baseline_submission.ipynb`.

In [ ]:
# Install dependencies (Kaggle has torch preinstalled, but we ensure ultralytics is present)
import sys, subprocess

def pip_install(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', *args]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

# Ultralytics YOLO (latest)
pip_install('ultralytics')

import os
from pathlib import Path
from ultralytics import YOLO

print('Ultralytics version:', YOLO.__module__)

## Configure dataset and paths

Update the paths below to point to your Kaggle input datasets.

### Examples
- If you attach a dataset named `visdrone-yolo` that contains:
  - `images/train/` and `images/val/`
  - `labels/train/` and `labels/val/`
  then the base directory on Kaggle is typically `/kaggle/input/visdrone-yolo`.

- For pretrained weights, if you upload a dataset named `zalo-pretrained-weights` with `yolov8n.pt`, you can refer to it as `/kaggle/input/zalo-pretrained-weights/yolov8n.pt`.

In [ ]:
# ==== USER CONFIGURATION (EDIT THIS CELL ON KAGGLE) ====

# Root directory of the YOLO-formatted dataset (images/ and labels/ subfolders)
DATA_ROOT = Path('/kaggle/input/visdrone-yolo')  # <-- CHANGE THIS to your dataset path

# Image folders
TRAIN_IMAGES_DIR = DATA_ROOT / 'images' / 'train'
VAL_IMAGES_DIR = DATA_ROOT / 'images' / 'val'

# Number of classes and their names for your dataset
# Example for VisDrone (10+ classes) – adjust to match your dataset
N_CLASSES = 10
CLASS_NAMES = [
    'pedestrian',
    'people',
    'bicycle',
    'car',
    'van',
    'truck',
    'tricycle',
    'awning-tricycle',
    'bus',
    'motor'
]  # <-- Adapt to your actual label set if needed

# Path to pretrained weights (starting point).
# Option 1: use a built-in YOLO checkpoint (e.g. 'yolov8n.pt').
PRETRAINED_WEIGHTS = 'yolov8n.pt'

# Option 2: if you upload your own weights as a Kaggle dataset, uncomment and point to them, e.g.:
# PRETRAINED_WEIGHTS = '/kaggle/input/zalo-pretrained-weights/yolov8n.pt'

# Output paths
WORK_DIR = Path('/kaggle/working')
DATA_YAML_PATH = WORK_DIR / 'zalo_yolo_data.yaml'

print('TRAIN_IMAGES_DIR:', TRAIN_IMAGES_DIR)
print('VAL_IMAGES_DIR:', VAL_IMAGES_DIR)
print('PRETRAINED_WEIGHTS:', PRETRAINED_WEIGHTS)
print('DATA_YAML_PATH:', DATA_YAML_PATH)

assert TRAIN_IMAGES_DIR.exists(), f'Train images dir not found: {TRAIN_IMAGES_DIR}'
assert VAL_IMAGES_DIR.exists(), f'Val images dir not found: {VAL_IMAGES_DIR}'

In [ ]:
# Create YOLO data config YAML in /kaggle/working/
data_yaml = (
    f"path: {DATA_ROOT.as_posix()}\n"
    f"train: {TRAIN_IMAGES_DIR.relative_to(DATA_ROOT).as_posix()}\n"
    f"val: {VAL_IMAGES_DIR.relative_to(DATA_ROOT).as_posix()}\n"
    f"\n"
    f"nc: {N_CLASSES}\n"
    f"names: {CLASS_NAMES}\n"
)

with open(DATA_YAML_PATH, 'w', encoding='utf-8') as f:
    f.write(data_yaml)

print('Wrote data YAML to:', DATA_YAML_PATH)
print('\n=== data.yaml ===')
print(data_yaml)

## Start finetuning

This cell launches YOLO training using the data configuration above.

Key hyperparameters you might want to adjust:
- `epochs`: number of training epochs (e.g. 30–100).
- `imgsz`: training image size (e.g. 640 or 896).
- `batch`: batch size (depends on GPU memory on Kaggle).
- `lr0`: initial learning rate.

The resulting weights (e.g. `best.pt`) will be saved under `/kaggle/working/runs/detect/<exp>/`.

In [ ]:
# Launch YOLO training / finetuning

model = YOLO(PRETRAINED_WEIGHTS)

results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=50,          # adjust based on time/resources
    imgsz=640,          # typical YOLOv8 size
    batch=16,           # tune based on available GPU memory
    lr0=1e-3,           # initial learning rate
    patience=10,        # early stopping patience
    optimizer='SGD',    # or 'AdamW'
    project=str(WORK_DIR / 'runs'),
    name='zalo_yolo_finetune',
    exist_ok=True,
)

results

## Locate and export trained weights

After training, the best model checkpoint is typically saved as:

- `/kaggle/working/runs/detect/zalo_yolo_finetune/weights/best.pt`

You can download this file from the Kaggle notebook output panel. Then, on your local machine, place it under `model/` and update your inference notebook (`baseline_submission.ipynb`) to use this new `best.pt` instead of `yolov8n.pt`.

In [ ]:
# Convenience: print expected best.pt path and check if it exists
best_path = WORK_DIR / 'runs' / 'detect' / 'zalo_yolo_finetune' / 'weights' / 'best.pt'
print('Expected best weights path:', best_path)
print('Exists:', best_path.exists())

# If desired, you could also copy or rename it here, e.g.:
# target = WORK_DIR / 'zalo_yolo_best.pt'
# if best_path.exists():
#     import shutil
#     shutil.copy(best_path, target)
#     print('Copied to:', target)